# Quiz Feedback Testing Notebook

Testing notebook for the `quiz_feedback` chain which generates encouraging feedback for quiz answers.

**Chain Function:** `generate_feedback(question, student_answer, correct_answer, is_correct)`

## 1. Setup

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Add parent directory to path for imports
sys.path.insert(0, os.path.abspath('../..'))

load_dotenv()
print("Environment loaded")

In [ ]:
from ai_chains.chains import quiz_feedback

print("Modules imported")

## 2. Chain Configuration

In [ ]:
# Display prompt template
with open('../../ai_chains/prompts/quiz_feedback.yaml', 'r', encoding='utf-8') as f:
    print("Prompt Template:")
    print(f.read())

In [ ]:
# Show LLM configuration
print("LLM Configuration:")
print(f"  OpenRouter Model: {os.getenv('OPENROUTER_MODEL', 'google/gemma-7b-it')}")
print(f"  Z.AI Model: {os.getenv('ZAI_MODEL', 'glm-4.7')}")
print(f"  OpenRouter Key: {'Set' if os.getenv('OPENROUTER_API_KEY') else 'Not set'}")
print(f"  Z.AI Key: {'Set' if os.getenv('ZAI_API_KEY') else 'Not set'}")

## 3. Test Cases

### Test 1: Correct Answer

In [ ]:
print("Test 1: Correct Answer")
print("="*50)

result = quiz_feedback.generate_feedback(
    question="Bagaimana cara membuat variabel baru di Python?",
    student_answer="nama = 'value'",
    correct_answer="nama = 'value'",
    is_correct=True
)

print("\nQuestion: Bagaimana cara membuat variabel baru di Python?")
print("Student Answer: nama = 'value'")
print("Correct Answer: nama = 'value'")
print("\nFeedback:")
print(result)

### Test 2: Incorrect Answer

In [ ]:
print("Test 2: Incorrect Answer")
print("="*50)

result = quiz_feedback.generate_feedback(
    question="Bagaimana cara membuat variabel baru di Python?",
    student_answer="var nama = 'value'",
    correct_answer="nama = 'value'",
    is_correct=False
)

print("\nQuestion: Bagaimana cara membuat variabel baru di Python?")
print("Student Answer: var nama = 'value'")
print("Correct Answer: nama = 'value'")
print("\nFeedback:")
print(result)

### Test 3: Multiple Choice Questions

In [ ]:
print("Test 3: Multiple Choice Questions")
print("="*50)

# Quiz questions from python_basics course
quiz_tests = [
    {
        "question": "Apa output dari print(type(3.14))?",
        "student_answer": "<class 'int'>",
        "correct_answer": "<class 'float'>",
        "is_correct": False
    },
    {
        "question": "Apa tipe data yang SELALU dikembalikan oleh input()?",
        "student_answer": "str",
        "correct_answer": "str",
        "is_correct": True
    }
]

for i, test in enumerate(quiz_tests, 1):
    result = quiz_feedback.generate_feedback(
        question=test["question"],
        student_answer=test["student_answer"],
        correct_answer=test["correct_answer"],
        is_correct=test["is_correct"]
    )
    
    status = "CORRECT" if test["is_correct"] else "INCORRECT"
    print(f"\n--- Quiz {i}: {status} ---")
    print(f"Q: {test['question']}")
    print(f"A: {test['student_answer']}")
    print(f"Feedback: {result[:200]}...")

### Test 4: Code Completion Questions

In [ ]:
print("Test 4: Code Completion Questions")
print("="*50)

# Code completion from quiz
result = quiz_feedback.generate_feedback(
    question="Lengkapi kode untuk mengkonversi string '100' menjadi integer:\nangka = ___('100')",
    student_answer="str",  # Wrong answer
    correct_answer="int",
    is_correct=False
)

print("\nQuestion: Lengkapi kode untuk mengkonversi string '100' menjadi integer")
print("Student Answer: str")
print("Correct Answer: int")
print("\nFeedback:")
print(result)

### Test 5: Edge Cases

In [ ]:
print("Test 5: Edge Cases")
print("="*50)

# Empty answer
print("\n5a. Empty student answer:")
result = quiz_feedback.generate_feedback(
    question="Apa itu variabel?",
    student_answer="",
    correct_answer="Wadah untuk menyimpan nilai",
    is_correct=False
)
print(f"Feedback: {result}")

# Numeric answer
print("\n5b. Numeric answer:")
result = quiz_feedback.generate_feedback(
    question="Apa hasil dari int(3.9)?",
    student_answer=3,
    correct_answer=3,
    is_correct=True
)
print(f"Feedback: {result}")

## 4. Analysis

In [ ]:
print("Analysis Summary")
print("="*50)

# Test both correct and incorrect scenarios
test_cases = [
    {"is_correct": True, "question": "Q1", "student": "A", "correct": "A"},
    {"is_correct": False, "question": "Q2", "student": "B", "correct": "A"},
    {"is_correct": True, "question": "Q3", "student": "C", "correct": "C"},
    {"is_correct": False, "question": "Q4", "student": "D", "correct": "C"},
]

correct_count = 0
for tc in test_cases:
    result = quiz_feedback.generate_feedback(
        question=tc["question"],
        student_answer=tc["student"],
        correct_answer=tc["correct"],
        is_correct=tc["is_correct"]
    )
    
    # Check if response is appropriate
    if tc["is_correct"]:
        # Should celebrate
        appropriate = any(word in result.lower() for word in ["bagus", "benar", "hebat", "excellent", "sempurna"])
    else:
        # Should be corrective but gentle
        appropriate = any(word in result.lower() for word in ["salah", "kurang", "seharusnya", "coba"])
    
    status = "OK" if appropriate else "CHECK"
    correct_count += 1 if appropriate else 0
    print(f"  [{status}] Test {'CORRECT' if tc['is_correct'] else 'INCORRECT'}: Appropriate tone")

print(f"\nQuality Score: {correct_count}/{len(test_cases)} appropriate responses")

## 5. Notes

**Expected Output for Correct Answer:**
- Celebrates success
- Explains WHY the answer is correct
- Encouraging tone in Indonesian

**Expected Output for Incorrect Answer:**
- Gentle correction
- Explains the misunderstanding
- Provides the correct answer
- Encouraging tone (no discouragement)

**Fallback Behavior:**
- Correct: `"Benar! Bagus sekali!"`
- Incorrect: `"Salah. Jawaban yang benar adalah: {correct_answer}"`